# Day 4 — Exercises: NumPy and pandas
## Master's Course: Quantitative Skills for Biology

**Topics covered:** NumPy arrays, vectorised operations, broadcasting, pandas DataFrames, `.loc`/`.iloc`, boolean indexing, `groupby`, `NaN`, `dropna()`, `fillna()`, loading real datasets.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns   # for loading real datasets

np.random.seed(42)

---
## Exercise 1 — NumPy Array Operations

Given the raw read counts below:
1. Apply a **log2 transformation** (add a pseudocount of 1 first: `log2(x + 1)`)
2. Compute mean, standard deviation, min, and max of the transformed values
3. Perform **min-max normalisation** so all values are in the range [0, 1]:
   `normalised = (x - x.min()) / (x.max() - x.min())`
4. Count how many transformed values are above the mean

In [ ]:
raw_counts = np.array([0, 145, 32, 980, 7, 1204, 56, 430, 3, 88])

# 1. Log2 transform with pseudocount
log_counts = 

# 2. Summary statistics

# 3. Min-max normalisation
normalised = 
print("Normalised:", normalised.round(3))

# 4. Count above-mean values


---
## Exercise 2 — Building and Inspecting a DataFrame

Create a DataFrame from the dictionary below. Then:
1. Print its shape, column names, and dtypes
2. Add a column `log2fc` = log2(treated / control), using a pseudocount of 0.1
3. Print a statistical summary using `.describe()`
4. How many genes have `log2fc > 1`?

In [ ]:
data = {
    "gene":    ["BRCA1", "TP53", "EGFR", "MYC", "PTEN", "KRAS", "AKT1", "CDK4"],
    "control": [8.43,    7.12,   9.01,   11.5,  5.30,   6.77,   8.90,   7.45 ],
    "treated": [10.2,    7.05,   9.44,   13.1,  5.28,   6.80,   9.10,   9.30 ],
    "pvalue":  [0.002,   0.810,  0.031,  0.001, 0.950,  0.880,  0.045,  0.003]
}

df = pd.DataFrame(data).set_index("gene")

# 1. Shape, columns, dtypes

# 2. log2fc column

# 3. Describe

# 4. How many genes have log2fc > 1?


---
## Exercise 3 — `.loc` and `.iloc` Selection

Using the DataFrame from Exercise 2 (make sure you ran that cell first):

1. Select the row for `"EGFR"` using `.loc`
2. Select the `control` and `treated` columns for the first 4 genes using `.iloc`
3. Use a **boolean mask** to select all genes where `pvalue < 0.05`
4. From those significant genes, print only the `log2fc` column sorted descending

In [ ]:
# 1. Row for EGFR

# 2. First 4 genes, control and treated columns (use .iloc)

# 3. Boolean mask: pvalue < 0.05
sig_mask = 
sig_genes = df[sig_mask]
print("Significant genes:")
print(sig_genes)

# 4. log2fc for significant genes, sorted descending


---
## Exercise 4 — `groupby` and Aggregation

Load the Titanic dataset and answer the following using `groupby`:

1. What is the **mean survival rate** for each passenger class (`pclass`)?
2. What is the **mean age** for survivors vs non-survivors (`survived`)?
3. What is the **median fare** paid by each sex (`sex`)?
4. Which embarkation port (`embark_town`) had the highest mean fare?

*Hint: `df.groupby('col')['col2'].mean()`*

In [ ]:
titanic = sns.load_dataset("titanic")
print("Shape:", titanic.shape)
titanic.head(3)

In [ ]:
# 1. Mean survival rate per class

# 2. Mean age for survivors vs non-survivors

# 3. Median fare by sex

# 4. Embarkation port with highest mean fare


---
## Exercise 5 — NaN Handling

Using the Titanic DataFrame:

1. Print the **count of missing values** per column
2. The `age` column has missing values. Fill them with the **median age** for each passenger class (i.e., class 1 passengers get the class 1 median, etc.)
3. Drop any rows where `embark_town` is missing
4. Confirm there are no more NaNs in `age` and `embark_town`

*Hint for 2: `df.groupby('pclass')['age'].transform('median')` returns the group median aligned to each row*

In [ ]:
titanic_clean = titanic.copy()   # work on a copy to preserve the original

# 1. Missing value counts

# 2. Fill age with per-class median
titanic_clean["age"] = 

# 3. Drop rows with missing embark_town

# 4. Confirm no more NaNs
print(titanic_clean[["age", "embark_town"]].isna().sum())

---
## Exercise 6 — Full DEG Pipeline

Recreate the synthetic expression dataset from the Day 4 notebook and build a complete **differential expression table**:

1. Generate the data (copy the setup code from Day 4, or use the code below)
2. Compute mean control, mean treated, and log2 fold-change per gene
3. Add a simulated p-value column (code provided)
4. Filter for **differentially expressed genes**: |log2fc| > 1 **and** pvalue < 0.05
5. Sort the result by |log2fc| descending
6. Print how many genes are upregulated vs downregulated

In [ ]:
np.random.seed(42)
n_genes = 20
gene_names   = [f"GENE_{i:03d}" for i in range(1, n_genes + 1)]
sample_names = [f"ctrl_{i}" for i in range(1, 4)] + [f"treat_{i}" for i in range(1, 4)]
ctrl_cols    = ["ctrl_1",  "ctrl_2",  "ctrl_3"]
treat_cols   = ["treat_1", "treat_2", "treat_3"]

ctrl_data    = np.random.normal(8.0, 1.0, (n_genes, 3))
treat_mean   = np.full(n_genes, 8.0); treat_mean[:5] = 10.0
treat_data   = np.random.normal(treat_mean[:, None], 1.0, (n_genes, 3))
df_deg       = pd.DataFrame(np.hstack([ctrl_data, treat_data]),
                            index=gene_names, columns=sample_names)
df_deg.index.name = "gene"

# Simulated p-values (smaller for first 5 genes)
np.random.seed(7)
df_deg["pvalue"] = np.where(
    df_deg.index.isin([f"GENE_{i:03d}" for i in range(1, 6)]),
    np.random.uniform(0.0001, 0.04, n_genes)[:5],
    np.random.uniform(0.05, 1.0, n_genes)[5:]
)

# 2. Add mean and log2fc columns

# 3. p-value column already added above

# 4. Filter for DEGs

# 5. Sort by |log2fc|

# 6. Count up vs downregulated
